# 04 — Frozen held-out evaluation / benchmark / demo

Implemented frozen-config test evaluation and real core benchmark. Requires a completed full-validation selection; Kaggle test execution evidence pending.


## Environment and assets

Enable a Kaggle GPU accelerator. Clone/mount this repository, then install its
`traffic-mot/requirements.txt` and `requirements-dev.txt` into an environment with
matching PyTorch/torchvision CUDA wheels. Inspect `pip check` before proceeding.
Clone the official RT-DETR repository at commit
`29320b6fd828f8e0987a71426cf2d961b09dfed7` into `RT-DETR/` if it is absent;
keep this checkout clean. Do not substitute a YOLO detector or auto-download weights.

Mount the real bundle (`best.pth`, `demo_config.yml`, `label_map.json`,
`classes.txt`, `bundle_manifest.json`). Edit paths below; REPLACE_ME must not remain.
Preflight rejects missing/wrong weights, mapping, preprocessing or source/GPU.
No notebook has been executed on Kaggle in this implementation session.

Repository: https://github.com/NguyenDoKhaiHoan/DeepLearning
Exact clone/install commands: `notebooks/kaggle/README.md` in that repository.
The Git repository contains source only; attach the checkpoint bundle and video
as Kaggle Input separately. Restart the kernel if dependency installation changed
an already imported PyTorch/NumPy version.


In [ ]:
from pathlib import Path
import sys

# Point to the checked-out repository and real Kaggle Input bundle.
REPO_ROOT = Path("/kaggle/working/DeepLearning")
PROJECT_ROOT = REPO_ROOT / "traffic-mot"
BUNDLE_DIR = Path("/kaggle/input/REPLACE_ME/E1_p2_final")
UPSTREAM_REPO = REPO_ROOT / "RT-DETR/rtdetrv2_pytorch"
VIDEO = Path("/kaggle/input/REPLACE_ME/traffic.mp4")
OUTPUT_ROOT = Path("/kaggle/working/traffic_mot_results")
MOT_DATA_ROOT = Path("/kaggle/input/visdrone-mot")  # Use the actual Add Input mount path.
MOT_MANIFEST = OUTPUT_ROOT / "mot_split_manifest.json"  # Generated below for notebooks 02-04.
assert (PROJECT_ROOT / "src/traffic_mot").is_dir(), "Mount/clone this repository first"
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from traffic_mot.utils.kaggle import (prepare_detection_config, check_video_dataset,
                                     check_mot_dataset, run_detection, run_stage)


In [ ]:
# VisDrone MOT uses image sequences; do not convert them into a fake-FPS MP4.
# Create from ZIPs or extracted directories; val is tuning, test-dev is held out.
from traffic_mot.utils.mot_data import prepare_mot_manifest
if not MOT_MANIFEST.exists():
    prepare_mot_manifest(MOT_MANIFEST, root=MOT_DATA_ROOT)
print("MOT_MANIFEST:", MOT_MANIFEST)
# Preparation lists frame/GT members without parsing held-out test annotations.
# Input preparation only; final test additionally requires the frozen validation artifact.


In [ ]:
config_path, preflight_report = prepare_detection_config(
    PROJECT_ROOT, BUNDLE_DIR, UPSTREAM_REPO, OUTPUT_ROOT / "configs",
    device="cuda:0", require_gpu=True)
print(preflight_report)


## Select the actual preceding validation run

Notebook03 writes into ablation_<uuid>, not necessarily a folder named ablation.
Set ABLATION_RUN below to that directory when running in a fresh kernel, or the
helper uses the existing notebook03 variable/unique frozen selection.
Smoke candidate_config.yaml cannot be renamed into a final configuration: source
manifest, full validation counts, selection checksum and tracker/model profiles
are verified before test GT or inference. No thresholds are tuned on test.


In [ ]:
from traffic_mot.evaluation.final import resolve_frozen_config
# Fresh kernel example (replace with your completed FULL validation run):
# ABLATION_RUN = OUTPUT_ROOT / "ablation_<uuid>"
FROZEN_CONFIG = resolve_frozen_config(OUTPUT_ROOT, globals().get("ABLATION_RUN"))
CONFIG_PATH = Path(globals().get("CONFIG_PATH", config_path))
assert CONFIG_PATH.is_file(), f"Missing detector config: {CONFIG_PATH}"
print("Frozen validation config:", FROZEN_CONFIG)
# No test GT is read until the final runner validates the freeze artifact.


In [ ]:
from uuid import uuid4
import json
FINAL_RUN = OUTPUT_ROOT / ("final_" + uuid4().hex)
arguments = [
    "--config", str(FROZEN_CONFIG), "--detector-config", str(CONFIG_PATH),
    "--manifest", str(MOT_MANIFEST), "--split", "test",
    "--output-root", str(FINAL_RUN), "--benchmark-frames", "100"]
# Optional demo playback (display-only FPS, not camera/processing FPS):
# arguments += ["--save-video", "--visualization-fps", "10"]
run_stage(PROJECT_ROOT, "final_evaluation.py", arguments)
result = json.loads((FINAL_RUN / "final_summary.json").read_text())
print(result)
print("DONE. Outputs:", FINAL_RUN)
# Benchmark measures actual decode+detector+tracking/OAR after warmup.
# Rendering/encoding/export are excluded; cached replay is never reported as core FPS.


## Acceptance evidence

Retain manifest, resolved config, source/checkpoint/data hashes, exact environment, raw predictions and decoded video verification. Only real runs on the declared split may populate metric tables. Missing assets/data/GPU are blockers, not zero scores.
